# 5. Clustering

**Machine Learning Fundamentals and Predictive Analytics — Notebook 5 of 11**

Everything so far has been **supervised**: you had labels, and the model learned to reproduce
them. **Clustering** is unsupervised — there are no labels, and the task is to find natural
groups in the data.

This changes the whole discipline. There is no accuracy to compute, no test set that settles
arguments. Clustering results have to be judged by internal measures, by stability, and above
all by **whether the groups are useful**.

### What you will learn

1. **K-Means**: the algorithm, its objective, and its failure modes
2. Choosing $k$: the **elbow method** and the **silhouette score**
3. Why initialisation matters, and what `k-means++` does
4. **Feature scaling** — mandatory again
5. **Hierarchical clustering** and dendrograms
6. **DBSCAN**: density-based clustering that finds outliers and non-convex shapes
7. **Gaussian Mixture Models**: soft, probabilistic clustering
8. Evaluation with and without ground truth
9. A customer-segmentation case study, end to end

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples, adjusted_rand_score,
                             normalized_mutual_info_score, calinski_harabasz_score,
                             davies_bouldin_score)
from sklearn.datasets import make_blobs, make_moons, make_circles
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

rng = np.random.default_rng(seed=5)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 5.1 K-Means

**Goal:** partition $n$ points into $k$ clusters so that the total within-cluster squared
distance to the cluster centre is as small as possible:

$$\min_{C_1,\dots,C_k}\ \sum_{j=1}^{k}\sum_{\mathbf{x}\in C_j}\|\mathbf{x} - \boldsymbol\mu_j\|^2$$

That quantity is called the **inertia** or **within-cluster sum of squares (WCSS)**.

**The algorithm (Lloyd's):**

1. Choose $k$ initial centroids
2. **Assign** each point to its nearest centroid
3. **Update** each centroid to the mean of its assigned points
4. Repeat 2–3 until nothing moves

Each step provably decreases the inertia, so it always converges — but only to a **local**
optimum. Finding the global optimum is NP-hard, which is why we restart from several
initialisations (`n_init`).

In [ ]:
# K-Means from scratch, so the loop is visible
def kmeans_from_scratch(X, k, n_iter=100, seed=0):
    g = np.random.default_rng(seed)
    centroids = X[g.choice(len(X), k, replace=False)]
    history = [centroids.copy()]
    labels = np.zeros(len(X), dtype=int)
    for _ in range(n_iter):
        d = np.sqrt(((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
        new_labels = d.argmin(axis=1)
        new_centroids = np.array([X[new_labels == j].mean(axis=0)
                                  if (new_labels == j).any() else centroids[j]
                                  for j in range(k)])
        if np.allclose(new_centroids, centroids) and (new_labels == labels).all():
            labels = new_labels
            break
        centroids, labels = new_centroids, new_labels
        history.append(centroids.copy())
    inertia = sum(((X[labels == j] - centroids[j]) ** 2).sum() for j in range(k))
    return labels, centroids, inertia, history

X_blob, y_blob = make_blobs(n_samples=400, centers=4, cluster_std=1.1, random_state=7)
labels_s, cent_s, inertia_s, hist = kmeans_from_scratch(X_blob, 4, seed=3)

km = KMeans(n_clusters=4, n_init=10, random_state=3).fit(X_blob)
print(f"From scratch  : inertia {inertia_s:.2f}, converged in {len(hist)} iterations")
print(f"scikit-learn  : inertia {km.inertia_:.2f}, {km.n_iter_} iterations")
print(f"Same partition (up to label permutation)? "
      f"{adjusted_rand_score(labels_s, km.labels_) > 0.99}")

In [ ]:
# Watch the centroids move
fig, axes = plt.subplots(1, min(5, len(hist)), figsize=(16, 3.4))
for step, ax in enumerate(np.atleast_1d(axes)):
    cents = hist[min(step, len(hist)-1)]
    d = np.sqrt(((X_blob[:, None, :] - cents[None, :, :]) ** 2).sum(axis=2))
    lab = d.argmin(axis=1)
    ax.scatter(X_blob[:, 0], X_blob[:, 1], c=lab, cmap="viridis", s=14, alpha=0.7)
    ax.scatter(cents[:, 0], cents[:, 1], c="crimson", s=180, marker="X",
               edgecolor="k", linewidth=1)
    ax.set_title(f"iteration {step}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Red crosses are the centroids. Assign, average, repeat -- that is the whole method.")

### What K-Means assumes

The objective (squared distance to a mean) bakes in strong assumptions:

- Clusters are **spherical** — because distance is isotropic
- Clusters have **similar sizes and densities** — because every point pulls its centroid equally
- Clusters are **convex and linearly separable**
- $k$ is **known in advance**
- Features are on **comparable scales**

When these hold, K-Means is fast and excellent. When they do not, it fails in predictable ways.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))

cases = []
# (a) ideal: spherical, equal-sized
Xa_, ya_ = make_blobs(n_samples=400, centers=3, cluster_std=1.0, random_state=1)
cases.append(("ideal spheres", Xa_, ya_, 3))
# (b) elongated (anisotropic)
Xb_, yb_ = make_blobs(n_samples=400, centers=3, random_state=1)
Xb_ = Xb_ @ np.array([[0.6, -0.63], [-0.4, 0.85]])
cases.append(("elongated clusters", Xb_, yb_, 3))
# (c) very different variances
Xc_, yc_ = make_blobs(n_samples=400, centers=3, cluster_std=[1.0, 3.0, 0.4], random_state=1)
cases.append(("unequal variances", Xc_, yc_, 3))
# (d) non-convex
Xd_, yd_ = make_moons(n_samples=400, noise=0.06, random_state=1)
cases.append(("non-convex (moons)", Xd_, yd_, 2))

for col, (name, Xs_, ys_, k_) in enumerate(cases):
    axes[0, col].scatter(Xs_[:, 0], Xs_[:, 1], c=ys_, cmap="viridis", s=12, alpha=0.75)
    axes[0, col].set_title(f"{name}\nTRUE groups", fontsize=9)
    kmm = KMeans(n_clusters=k_, n_init=10, random_state=0).fit(Xs_)
    axes[1, col].scatter(Xs_[:, 0], Xs_[:, 1], c=kmm.labels_, cmap="viridis", s=12, alpha=0.75)
    axes[1, col].scatter(kmm.cluster_centers_[:, 0], kmm.cluster_centers_[:, 1],
                         c="crimson", s=140, marker="X", edgecolor="k")
    axes[1, col].set_title(f"K-Means (ARI = {adjusted_rand_score(ys_, kmm.labels_):.3f})",
                           fontsize=9)
    for r in (0, 1):
        axes[r, col].set_xticks([]); axes[r, col].set_yticks([])
plt.tight_layout(); plt.show()

print("K-Means recovers the spheres perfectly and fails on the other three, each in a")
print("different way: it cuts elongated clusters across their length, it splits the large")
print("cluster to 'help' the small one, and it slices the moons straight down the middle.")

---
## 5.2 Initialisation: why `k-means++` matters

Random initial centroids can land badly, and the algorithm then converges to a poor local
optimum. **k-means++** spreads the initial centroids out: the first is random, and each
subsequent one is chosen with probability proportional to its squared distance from the
nearest existing centroid.

Combined with `n_init` (multiple restarts, keep the best inertia), this makes K-Means
reliable in practice. `n_init="auto"` is the modern default; set it explicitly if you want
reproducibility across versions.

In [ ]:
X_hard, y_hard = make_blobs(n_samples=500, centers=6, cluster_std=0.85, random_state=42)

random_inertias, pp_inertias = [], []
for seed in range(60):
    random_inertias.append(KMeans(6, init="random", n_init=1, random_state=seed)
                           .fit(X_hard).inertia_)
    pp_inertias.append(KMeans(6, init="k-means++", n_init=1, random_state=seed)
                       .fit(X_hard).inertia_)

plt.hist(random_inertias, bins=25, alpha=0.65, color="crimson", label="init='random'")
plt.hist(pp_inertias, bins=25, alpha=0.65, color="steelblue", label="init='k-means++'")
plt.xlabel("final inertia (lower is better)"); plt.ylabel("frequency")
plt.title("60 single-start runs: k-means++ almost never lands badly")
plt.legend(fontsize=8); plt.show()

print(f"random init  : best {min(random_inertias):.1f}, worst {max(random_inertias):.1f}, "
      f"mean {np.mean(random_inertias):.1f}")
print(f"k-means++    : best {min(pp_inertias):.1f}, worst {max(pp_inertias):.1f}, "
      f"mean {np.mean(pp_inertias):.1f}")
print(f"\nWith n_init=10 (the practical setting), the best of 10 restarts is taken:")
print(f"  inertia = {KMeans(6, n_init=10, random_state=0).fit(X_hard).inertia_:.1f}")

---
## 5.3 Choosing k

There is no correct answer — $k$ is a modelling choice. Three tools, to be used together.

### The elbow method

Plot inertia against $k$. Inertia always falls (with $k = n$ it reaches 0), so you look for
the **elbow**: the point after which extra clusters buy little. It is a judgement call, and
often ambiguous.

### The silhouette score

For each point $i$: let $a_i$ be its mean distance to other points in its own cluster, and
$b_i$ the mean distance to the points of the nearest *other* cluster. Then

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)} \in [-1, 1]$$

- $s_i \approx 1$ — comfortably inside its cluster
- $s_i \approx 0$ — on the boundary
- $s_i < 0$ — probably in the wrong cluster

The **silhouette score** is the mean over all points. Unlike inertia it does not automatically
improve with $k$, so you can maximise it.

### Two more indices

- **Calinski-Harabasz** — between-cluster over within-cluster dispersion; higher is better
- **Davies-Bouldin** — average similarity between each cluster and its most similar one;
  **lower** is better

In [ ]:
X_k, y_k = make_blobs(n_samples=600, centers=5, cluster_std=1.0, random_state=11)

ks = range(2, 13)
rows = []
for k in ks:
    km_ = KMeans(k, n_init=10, random_state=0).fit(X_k)
    rows.append({"k": k, "inertia": km_.inertia_,
                 "silhouette": silhouette_score(X_k, km_.labels_),
                 "calinski_harabasz": calinski_harabasz_score(X_k, km_.labels_),
                 "davies_bouldin": davies_bouldin_score(X_k, km_.labels_)})
diag = pd.DataFrame(rows)
print(diag.round(4).to_string(index=False))
print(f"\nTrue number of clusters: 5")
print(f"Best silhouette at k = {int(diag.loc[diag.silhouette.idxmax(), 'k'])}")
print(f"Best Calinski-Harabasz at k = {int(diag.loc[diag.calinski_harabasz.idxmax(), 'k'])}")
print(f"Best Davies-Bouldin at k = {int(diag.loc[diag.davies_bouldin.idxmin(), 'k'])}")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
ax[0].plot(diag.k, diag.inertia, "o-", color="steelblue")
ax[0].axvline(5, color="crimson", ls="--")
ax[0].set_xlabel("k"); ax[0].set_ylabel("inertia"); ax[0].set_title("Elbow plot", fontsize=10)

ax[1].plot(diag.k, diag.silhouette, "o-", color="seagreen")
ax[1].axvline(diag.loc[diag.silhouette.idxmax(), "k"], color="crimson", ls="--")
ax[1].set_xlabel("k"); ax[1].set_title("Silhouette (higher better)", fontsize=10)

ax[2].plot(diag.k, diag.calinski_harabasz, "o-", color="darkorange")
ax[2].set_xlabel("k"); ax[2].set_title("Calinski-Harabasz (higher)", fontsize=10)

ax[3].plot(diag.k, diag.davies_bouldin, "o-", color="purple")
ax[3].set_xlabel("k"); ax[3].set_title("Davies-Bouldin (lower)", fontsize=10)
plt.tight_layout(); plt.show()

# The elbow, made quantitative: distance from the line joining the endpoints
pts = np.column_stack([diag.k, diag.inertia / diag.inertia.max()])
line = pts[-1] - pts[0]
line = line / np.linalg.norm(line)
proj = pts[0] + np.outer((pts - pts[0]) @ line, line)
elbow_k = int(diag.k.iloc[int(np.argmax(np.linalg.norm(pts - proj, axis=1)))])
print(f"Elbow located automatically (max distance from the chord): k = {elbow_k}")
print("The elbow is often ambiguous; silhouette gives a number you can maximise.")

In [ ]:
# Silhouette PLOTS are more informative than the single average
def silhouette_plot(X, k, ax):
    km_ = KMeans(k, n_init=10, random_state=0).fit(X)
    labels = km_.labels_
    sil = silhouette_samples(X, labels)
    avg = sil.mean()
    y_low = 10
    for j in range(k):
        vals = np.sort(sil[labels == j])
        y_up = y_low + len(vals)
        ax.fill_betweenx(np.arange(y_low, y_up), 0, vals,
                         color=plt.cm.viridis(j / k), alpha=0.8)
        ax.text(-0.05, y_low + 0.5*len(vals), str(j), fontsize=8)
        y_low = y_up + 10
    ax.axvline(avg, color="crimson", ls="--")
    ax.set_title(f"k = {k}, mean silhouette = {avg:.3f}", fontsize=9)
    ax.set_xlabel("silhouette value"); ax.set_yticks([])
    ax.set_xlim(-0.2, 1)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, k in zip(axes, [2, 4, 5, 8]):
    silhouette_plot(X_k, k, ax)
plt.tight_layout(); plt.show()

print("What to look for in a silhouette plot:")
print("  * every cluster should reach above the mean line (red dashes)")
print("  * clusters should be of comparable width -- one thin sliver suggests over-splitting")
print("  * negative values are points assigned to the wrong cluster")
print("At k=8 several clusters are thin and dip below the average: k is too large.")

---
## 5.4 Scaling matters (again)

K-Means minimises squared Euclidean distance, so — exactly as with KNN — a feature measured in
larger units dominates the objective. **Always scale before clustering**, unless the units are
already comparable and you have a reason to keep them.

Use `RobustScaler` when features are skewed with outliers, which is the norm for monetary
variables.

In [ ]:
# Customers with three features on very different scales
m = 600
segments = rng.integers(0, 3, m)
annual_spend = np.array([30_000, 120_000, 400_000])[segments] * rng.lognormal(0, 0.25, m)
visits = np.array([4, 14, 30])[segments] + rng.normal(0, 2, m)
tenure_yr = np.array([1.0, 3.5, 7.0])[segments] + rng.normal(0, 0.7, m)

cust = pd.DataFrame({"annual_spend": annual_spend, "visits": visits, "tenure_yr": tenure_yr})
print(cust.describe().round(1).to_string())

raw_km = KMeans(3, n_init=10, random_state=0).fit(cust)
scaled_km = make_pipeline(StandardScaler(), KMeans(3, n_init=10, random_state=0)).fit(cust)

print(f"\nAgreement with the true segments (adjusted Rand index):")
print(f"  unscaled : {adjusted_rand_score(segments, raw_km.labels_):.4f}")
print(f"  scaled   : {adjusted_rand_score(segments, scaled_km[-1].labels_):.4f}")
print("\nUnscaled, the clustering is essentially a split on annual_spend alone --")
print("visits and tenure contribute nothing next to a 400,000-unit range.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(cust.annual_spend, cust.visits, c=segments, cmap="viridis", s=14)
ax[0].set_title("True segments"); ax[0].set_xlabel("annual spend"); ax[0].set_ylabel("visits")
ax[1].scatter(cust.annual_spend, cust.visits, c=raw_km.labels_, cmap="viridis", s=14)
ax[1].set_title(f"K-Means, unscaled (ARI {adjusted_rand_score(segments, raw_km.labels_):.3f})")
ax[1].set_xlabel("annual spend")
ax[2].scatter(cust.annual_spend, cust.visits, c=scaled_km[-1].labels_, cmap="viridis", s=14)
ax[2].set_title(f"K-Means, scaled (ARI {adjusted_rand_score(segments, scaled_km[-1].labels_):.3f})")
ax[2].set_xlabel("annual spend")
plt.tight_layout(); plt.show()

---
## 5.5 Hierarchical clustering

Instead of committing to $k$, build a **tree**. **Agglomerative** (bottom-up) clustering starts
with every point as its own cluster and repeatedly merges the two closest, until one cluster
remains. The merge history is a **dendrogram**, which you cut at whatever height you like.

**Linkage** determines what "closest" means between two clusters:

| Linkage | Distance between clusters | Tends to produce |
|---|---|---|
| **single** | closest pair | Long chains; can find non-convex shapes; sensitive to noise |
| **complete** | farthest pair | Compact, roughly equal-diameter clusters |
| **average** | mean over all pairs | A compromise |
| **Ward** | the merge that increases within-cluster variance least | Spherical, balanced clusters (the usual default) |

Advantages: no need to fix $k$ up front, the dendrogram is genuinely informative, and it works
with any distance matrix. Disadvantage: $O(n^2)$ memory and $O(n^3)$ time in general — not for
millions of rows.

In [ ]:
X_h, y_h = make_blobs(n_samples=60, centers=4, cluster_std=0.9, random_state=3)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, method in zip(axes, ["single", "complete", "average", "ward"]):
    Z = linkage(X_h, method=method)
    dendrogram(Z, ax=ax, no_labels=True, color_threshold=0.7*Z[:, 2].max())
    ax.set_title(f"linkage = '{method}'", fontsize=10)
    ax.set_ylabel("merge distance")
plt.tight_layout(); plt.show()

print("Read a dendrogram bottom-up: each horizontal bar is a merge, and its HEIGHT is how")
print("dissimilar the two merged clusters were. Long vertical gaps mean a natural split.")
print("Cutting the tree at a height gives you a clustering.")

In [ ]:
Z_ward = linkage(X_h, method="ward")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
dendrogram(Z_ward, ax=axes[0], no_labels=True, color_threshold=8)
axes[0].axhline(8, color="crimson", ls="--", lw=2)
axes[0].set_title("Ward dendrogram, cut at height 8", fontsize=10)

for ax, k in zip(axes[1:], [2, 4]):
    lab = fcluster(Z_ward, t=k, criterion="maxclust")
    ax.scatter(X_h[:, 0], X_h[:, 1], c=lab, cmap="viridis", s=45, edgecolor="k", linewidth=0.3)
    ax.set_title(f"cut to give {k} clusters (ARI vs truth "
                 f"{adjusted_rand_score(y_h, lab):.3f})", fontsize=10)
plt.tight_layout(); plt.show()

# The scikit-learn estimator
for k in (2, 3, 4, 5):
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward").fit(X_h)
    print(f"  k={k}: silhouette {silhouette_score(X_h, agg.labels_):.4f}, "
          f"ARI vs truth {adjusted_rand_score(y_h, agg.labels_):.4f}")

In [ ]:
# Where single linkage wins: chained, non-convex structure
X_c, y_c = make_circles(n_samples=400, noise=0.05, factor=0.45, random_state=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
axes[0].scatter(X_c[:, 0], X_c[:, 1], c=y_c, cmap="viridis", s=12)
axes[0].set_title("True structure: two rings", fontsize=9)
for ax, method in zip(axes[1:], ["ward", "complete", "single"]):
    lab = AgglomerativeClustering(n_clusters=2, linkage=method).fit_predict(X_c)
    ax.scatter(X_c[:, 0], X_c[:, 1], c=lab, cmap="viridis", s=12)
    ax.set_title(f"linkage='{method}'\nARI {adjusted_rand_score(y_c, lab):.3f}", fontsize=9)
for a_ in axes:
    a_.set_xticks([]); a_.set_yticks([]); a_.set_aspect("equal")
plt.tight_layout(); plt.show()

print("Single linkage recovers the rings perfectly, because it only ever needs one close")
print("pair to keep a chain going. Ward and complete linkage insist on compactness and")
print("cut the rings in half -- the same failure as K-Means.")
print("\nThe price of single linkage: one noisy point bridging two clusters merges them.")

---
## 5.6 DBSCAN: density-based clustering

DBSCAN takes a different view: a cluster is a **dense region**, and points in sparse regions
are **noise**. Two parameters:

- **`eps`** — the neighbourhood radius
- **`min_samples`** — how many points must be within `eps` for a point to be a *core point*

Point types:
- **Core** — has at least `min_samples` neighbours within `eps`
- **Border** — within `eps` of a core point, but not itself core
- **Noise** — neither; labelled `-1`

**Advantages:** finds arbitrarily shaped clusters, decides the number of clusters itself, and
identifies outliers explicitly.
**Disadvantages:** very sensitive to `eps`; struggles when clusters have very different
densities; and, being distance-based, suffers in high dimensions.

A practical way to choose `eps`: plot the sorted distance to each point's $k$-th nearest
neighbour and look for the knee.

In [ ]:
X_moon, y_moon = make_moons(n_samples=500, noise=0.07, random_state=1)
X_moon = np.vstack([X_moon, rng.uniform(-1.8, 2.8, (25, 2))])      # add some noise points
y_moon = np.r_[y_moon, np.full(25, -1)]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap="viridis", s=14)
axes[0].set_title("Truth: 2 moons + scattered noise", fontsize=9)

km_m = KMeans(2, n_init=10, random_state=0).fit(X_moon)
axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=km_m.labels_, cmap="viridis", s=14)
axes[1].set_title(f"K-Means, k=2\nARI {adjusted_rand_score(y_moon, km_m.labels_):.3f}",
                  fontsize=9)

for ax, eps in zip(axes[2:], [0.18, 0.30]):
    db = DBSCAN(eps=eps, min_samples=5).fit(X_moon)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise = (db.labels_ == -1).sum()
    ax.scatter(X_moon[:, 0], X_moon[:, 1], c=db.labels_, cmap="viridis", s=14)
    ax.set_title(f"DBSCAN eps={eps}\n{n_clusters} clusters, {n_noise} noise, "
                 f"ARI {adjusted_rand_score(y_moon, db.labels_):.3f}", fontsize=9)
for a_ in axes:
    a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

print("DBSCAN separates the moons AND flags the scattered points as noise (label -1),")
print("neither of which K-Means can do.")

In [ ]:
# Choosing eps with a k-distance plot
from sklearn.neighbors import NearestNeighbors
k_dist = NearestNeighbors(n_neighbors=5).fit(X_moon).kneighbors(X_moon)[0][:, -1]
k_dist_sorted = np.sort(k_dist)

plt.plot(k_dist_sorted, color="steelblue", lw=2)
plt.axhline(0.18, color="crimson", ls="--", label="eps = 0.18")
plt.xlabel("points, sorted"); plt.ylabel("distance to 5th nearest neighbour")
plt.title("k-distance plot: the knee suggests eps")
plt.legend(fontsize=8); plt.show()

print("The curve is flat for points inside dense clusters and rises sharply for outliers.")
print("The knee is a good starting value for eps.\n")

print(f"{'eps':>7}{'min_samples':>13}{'clusters':>10}{'noise':>8}{'silhouette':>12}")
for eps in (0.10, 0.15, 0.20, 0.30, 0.50):
    for ms in (5, 10):
        db = DBSCAN(eps=eps, min_samples=ms).fit(X_moon)
        lab = db.labels_
        n_cl = len(set(lab)) - (1 if -1 in lab else 0)
        mask = lab != -1
        sil = (silhouette_score(X_moon[mask], lab[mask])
               if n_cl > 1 and mask.sum() > n_cl else np.nan)
        print(f"{eps:>7.2f}{ms:>13}{n_cl:>10}{(lab == -1).sum():>8}{sil:>12.4f}")
print("\nDBSCAN is very sensitive to eps: 0.10 fragments the data, 0.50 merges everything.")
print("Note that silhouette should be computed on the non-noise points only.")

In [ ]:
# Where DBSCAN struggles: clusters of very different density
X_var = np.vstack([
    rng.normal([0, 0], 0.25, (200, 2)),        # dense
    rng.normal([4, 4], 1.20, (200, 2)),        # sparse
])
y_var = np.r_[np.zeros(200), np.ones(200)]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
axes[0].scatter(X_var[:, 0], X_var[:, 1], c=y_var, cmap="viridis", s=14)
axes[0].set_title("Truth: one dense, one sparse", fontsize=9)
for ax, eps in zip(axes[1:], [0.25, 0.6, 1.2]):
    db = DBSCAN(eps=eps, min_samples=5).fit(X_var)
    n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    ax.scatter(X_var[:, 0], X_var[:, 1], c=db.labels_, cmap="viridis", s=14)
    ax.set_title(f"eps={eps}: {n_cl} clusters, {(db.labels_==-1).sum()} noise\n"
                 f"ARI {adjusted_rand_score(y_var, db.labels_):.3f}", fontsize=9)
for a_ in axes:
    a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

print("No single eps works: small eps calls the sparse cluster noise, large eps merges both.")
print("This is DBSCAN's core limitation. HDBSCAN (a separate package) fixes it by varying")
print("the density threshold across the space.")

---
## 5.7 Gaussian Mixture Models: soft clustering

A **GMM** models the data as a mixture of $k$ Gaussian distributions:

$$p(\mathbf{x}) = \sum_{j=1}^{k}\pi_j\,\mathcal{N}(\mathbf{x}\mid\boldsymbol\mu_j, \Sigma_j)$$

Fitted by **Expectation-Maximisation**, which is K-Means' probabilistic cousin: E-step assigns
soft responsibilities, M-step updates means, covariances and weights.

Advantages over K-Means:

- **Soft assignments** — each point gets a probability of belonging to each cluster
- **Elliptical clusters** — the covariance $\Sigma_j$ can be stretched and rotated
- It is a proper **generative model**, so you can compute likelihoods, sample new data, and
  use **AIC/BIC** to choose $k$ in a principled way

The `covariance_type` parameter controls flexibility: `spherical` ≈ K-Means, through
`diag` and `tied`, up to `full` (each cluster gets its own arbitrary ellipse).

In [ ]:
# GMM handles elongated clusters that defeat K-Means
X_e, y_e = make_blobs(n_samples=500, centers=3, cluster_std=1.0, random_state=1)
X_e = X_e @ np.array([[0.6, -0.63], [-0.4, 0.85]])

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
axes[0].scatter(X_e[:, 0], X_e[:, 1], c=y_e, cmap="viridis", s=12)
axes[0].set_title("True groups (elongated)", fontsize=9)

km_e = KMeans(3, n_init=10, random_state=0).fit(X_e)
axes[1].scatter(X_e[:, 0], X_e[:, 1], c=km_e.labels_, cmap="viridis", s=12)
axes[1].set_title(f"K-Means\nARI {adjusted_rand_score(y_e, km_e.labels_):.3f}", fontsize=9)

for ax, cov in zip(axes[2:], ["spherical", "full"]):
    gm = GaussianMixture(3, covariance_type=cov, random_state=0, n_init=5).fit(X_e)
    lab = gm.predict(X_e)
    ax.scatter(X_e[:, 0], X_e[:, 1], c=lab, cmap="viridis", s=12)
    ax.set_title(f"GMM, covariance='{cov}'\nARI {adjusted_rand_score(y_e, lab):.3f}",
                 fontsize=9)
for a_ in axes:
    a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

print("With full covariance the GMM fits rotated ellipses and recovers the true groups.")
print("With spherical covariance it reduces to something very close to K-Means.")

In [ ]:
# Soft assignments, and the points the model is unsure about
gm = GaussianMixture(3, covariance_type="full", random_state=0, n_init=5).fit(X_e)
proba = gm.predict_proba(X_e)
confidence = proba.max(axis=1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
sc = ax[0].scatter(X_e[:, 0], X_e[:, 1], c=confidence, cmap="RdYlGn", s=18, vmin=0.33, vmax=1)
plt.colorbar(sc, ax=ax[0], label="confidence of the top assignment")
ax[0].set_title("Red points sit between clusters")
ax[1].hist(confidence, bins=40, color="steelblue")
ax[1].set_xlabel("max posterior probability"); ax[1].set_ylabel("count")
ax[1].set_title("Most points are assigned confidently")
plt.tight_layout(); plt.show()

print("Soft assignments for the three least confident points:")
uncertain = np.argsort(confidence)[:3]
print(pd.DataFrame(proba[uncertain].round(3),
                   columns=[f"cluster {j}" for j in range(3)],
                   index=[f"point {i}" for i in uncertain]))
print("\nThis is genuinely useful: you can exclude ambiguous customers from a campaign,")
print("or route them for manual review, instead of forcing a hard label.")

In [ ]:
# Choosing k with AIC and BIC -- a principled alternative to the elbow
X_bic, y_bic = make_blobs(n_samples=700, centers=5, cluster_std=1.0, random_state=11)
ks = range(1, 13)
aics, bics, sils = [], [], []
for k in ks:
    gm_ = GaussianMixture(k, covariance_type="full", random_state=0, n_init=3).fit(X_bic)
    aics.append(gm_.aic(X_bic)); bics.append(gm_.bic(X_bic))
    sils.append(silhouette_score(X_bic, gm_.predict(X_bic)) if k > 1 else np.nan)

plt.plot(list(ks), aics, "o-", color="steelblue", label="AIC")
plt.plot(list(ks), bics, "o-", color="crimson", label="BIC")
plt.axvline(5, color="black", ls="--", label="true k = 5")
plt.xlabel("number of components"); plt.ylabel("information criterion (lower is better)")
plt.title("BIC penalises extra components more than AIC")
plt.legend(fontsize=8); plt.show()

print(f"AIC minimised at k = {list(ks)[int(np.argmin(aics))]}")
print(f"BIC minimised at k = {list(ks)[int(np.argmin(bics))]}")
print("\nBIC is usually the better guide for choosing k: it penalises complexity more")
print("heavily and is consistent, meaning it finds the true k as n grows.")

---
## 5.8 Evaluating a clustering

**Without ground truth (internal measures)** — measure how compact and separated the clusters
are: silhouette, Calinski-Harabasz, Davies-Bouldin. These all reward spherical, well-separated
clusters, so they are biased *toward* what K-Means produces. Use them, but do not trust them
to adjudicate between K-Means and DBSCAN.

**With ground truth (external measures)** — available in benchmarks, and occasionally in
practice when a labelled subset exists:

| Measure | Range | Notes |
|---|---|---|
| **Adjusted Rand Index (ARI)** | −1 to 1 | Agreement of pairwise co-membership, corrected for chance. 0 = random |
| **Normalised Mutual Information (NMI)** | 0 to 1 | Shared information between the two partitions |
| **Homogeneity / completeness / V-measure** | 0 to 1 | Each cluster has one class / each class is in one cluster / their harmonic mean |

**The tests that matter most in practice:**

1. **Stability** — cluster a bootstrap resample. Do the same groups appear?
2. **Interpretability** — can you describe each cluster in a sentence a colleague accepts?
3. **Actionability** — does the segmentation change what you would *do*?

In [ ]:
# Internal measures are biased toward spherical clusters
print("Two algorithms on the moons data, judged two ways:\n")
X_mm, y_mm = make_moons(n_samples=400, noise=0.06, random_state=1)
for name, lab in [("K-Means (k=2)", KMeans(2, n_init=10, random_state=0).fit_predict(X_mm)),
                  ("DBSCAN", DBSCAN(eps=0.25, min_samples=5).fit_predict(X_mm))]:
    mask = lab != -1
    print(f"  {name:<16} silhouette {silhouette_score(X_mm[mask], lab[mask]):>7.4f}   "
          f"ARI vs truth {adjusted_rand_score(y_mm, lab):>7.4f}")
print("\nK-Means wins on silhouette and loses badly on ARI. Silhouette rewards compactness,")
print("and the true moons are not compact. Internal measures encode an assumption -- know")
print("which assumption before you optimise one.")

In [ ]:
# Stability: the most useful label-free check
def cluster_stability(X, k, algo="kmeans", n_boot=40, frac=0.85, seed=0):
    '''Mean ARI between clusterings of overlapping subsamples.'''
    g = np.random.default_rng(seed)
    n_ = len(X)
    scores = []
    for _ in range(n_boot):
        i1 = g.choice(n_, int(frac*n_), replace=False)
        i2 = g.choice(n_, int(frac*n_), replace=False)
        common = np.intersect1d(i1, i2)
        if len(common) < 20:
            continue
        def fit(idx):
            Xs_ = StandardScaler().fit_transform(X[idx])
            if algo == "kmeans":
                return KMeans(k, n_init=10, random_state=0).fit_predict(Xs_)
            return AgglomerativeClustering(n_clusters=k).fit_predict(Xs_)
        l1 = pd.Series(fit(i1), index=i1)
        l2 = pd.Series(fit(i2), index=i2)
        scores.append(adjusted_rand_score(l1.loc[common], l2.loc[common]))
    return np.mean(scores), np.std(scores)

print("Stability of K-Means on data that really has 5 clusters:")
print(f"{'k':>4}{'mean ARI':>12}{'sd':>8}")
for k in range(2, 10):
    mu, sd = cluster_stability(X_bic, k)
    flag = "  <- most stable" if k == 5 else ""
    print(f"{k:>4}{mu:>12.4f}{sd:>8.4f}{flag}")
print("\nStability peaks at the true k. This is often a better guide than the elbow,")
print("and it needs no labels -- just two overlapping subsamples and an ARI.")

---
## 5.9 Case study: customer segmentation

The full workflow on realistic RFM-style data (Recency, Frequency, Monetary), which is the
standard starting point for retail segmentation.

In [ ]:
m_c = 1_200
kind = rng.choice(["champion", "loyal", "at_risk", "new"], m_c, p=[0.15, 0.3, 0.3, 0.25])
params = {
    "champion": (5, 25, 12.2),      # (recency days-ish, frequency, log spend)
    "loyal":    (25, 12, 11.3),
    "at_risk":  (150, 6, 11.0),
    "new":      (15, 2, 10.4),
}
recency = np.array([params[k_][0] for k_ in kind]) * rng.lognormal(0, 0.4, m_c)
frequency = np.maximum(1, np.array([params[k_][1] for k_ in kind]) + rng.normal(0, 3, m_c))
monetary = np.exp(np.array([params[k_][2] for k_ in kind]) + rng.normal(0, 0.4, m_c))
tenure_days = np.where(kind == "new", rng.uniform(5, 90, m_c), rng.uniform(200, 1800, m_c))

rfm = pd.DataFrame({"recency_days": recency, "frequency": frequency,
                    "monetary": monetary, "tenure_days": tenure_days})
print(rfm.describe().round(1).to_string())
print(f"\nSkewness: {rfm.skew().round(2).to_dict()}")
print("\nAll three RFM variables are right-skewed -- log them before clustering, or the")
print("distance metric will be dominated by a handful of whales.")

In [ ]:
# Step 1: transform and scale
work = pd.DataFrame({
    "log_recency": np.log1p(rfm.recency_days),
    "log_frequency": np.log1p(rfm.frequency),
    "log_monetary": np.log(rfm.monetary),
    "log_tenure": np.log(rfm.tenure_days),
})
Z = StandardScaler().fit_transform(work)

# Step 2: choose k using three signals at once
rows = []
for k in range(2, 9):
    km_ = KMeans(k, n_init=10, random_state=0).fit(Z)
    stab, _ = cluster_stability(work.to_numpy(), k, n_boot=20)
    rows.append({"k": k, "inertia": km_.inertia_,
                 "silhouette": silhouette_score(Z, km_.labels_),
                 "davies_bouldin": davies_bouldin_score(Z, km_.labels_),
                 "stability_ARI": stab})
sel = pd.DataFrame(rows)
print(sel.round(4).to_string(index=False))
print(f"\nSilhouette favours k = {int(sel.loc[sel.silhouette.idxmax(),'k'])}, "
      f"stability favours k = {int(sel.loc[sel.stability_ARI.idxmax(),'k'])}")
print("True number of underlying customer types: 4")

In [ ]:
# Step 3: fit the chosen model and PROFILE the clusters -- this is the real deliverable
K = 4
final = KMeans(K, n_init=20, random_state=0).fit(Z)
rfm["cluster"] = final.labels_

profile = rfm.groupby("cluster").agg(
    n=("monetary", "size"),
    recency_days=("recency_days", "median"),
    frequency=("frequency", "median"),
    monetary=("monetary", "median"),
    tenure_days=("tenure_days", "median"),
).round(1)
profile["share_%"] = (profile.n / len(rfm) * 100).round(1)
profile["revenue_share_%"] = (rfm.groupby("cluster").monetary.sum()
                              / rfm.monetary.sum() * 100).round(1)
print(profile.to_string())
print(f"\nAgreement with the true customer types: "
      f"ARI = {adjusted_rand_score(kind, final.labels_):.4f}")

In [ ]:
# Step 4: visualise and name the segments
overall = work.mean()
centres = pd.DataFrame(StandardScaler().fit(work).inverse_transform(final.cluster_centers_),
                       columns=work.columns)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
sns.heatmap((centres - overall).T, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[0])
ax[0].set_xlabel("cluster"); ax[0].set_title("Cluster centres vs the overall average")

pca = PCA(n_components=2).fit(Z)
P = pca.transform(Z)
sc = ax[1].scatter(P[:, 0], P[:, 1], c=final.labels_, cmap="viridis", s=12, alpha=0.75)
ax[1].scatter(*pca.transform(final.cluster_centers_).T, c="crimson", s=180, marker="X",
              edgecolor="k")
ax[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} of variance)")
ax[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax[1].set_title("Clusters projected onto two principal components")
plt.tight_layout(); plt.show()

print("Naming the segments from the profile table (this is the part that gets used):")
for c in range(K):
    p = profile.loc[c]
    if p.recency_days < 30 and p.frequency > 15:
        name, action = "CHAMPIONS", "reward, ask for referrals, protect at all costs"
    elif p.recency_days > 80:
        name, action = "AT RISK", "win-back campaign, discount, survey why they stopped"
    elif p.tenure_days < 150:
        name, action = "NEW", "onboarding sequence, drive a second purchase"
    else:
        name, action = "LOYAL", "cross-sell, loyalty tier, steady nurture"
    print(f"  Cluster {c} -> {name:<10} ({p['share_%']:.0f}% of customers, "
          f"{p['revenue_share_%']:.0f}% of revenue)")
    print(f"              action: {action}")
print("\nA segmentation that does not end in a table like this has not finished.")

---
## 5.10 Choosing an algorithm

```
Do you know how many clusters you want?
  YES, and clusters look round and similar-sized ......... K-Means
  YES, and clusters are elongated or overlapping ......... GMM (covariance='full')
  YES, and you want a hierarchy to inspect ............... Agglomerative (Ward)
  NO, and clusters are dense blobs with noise/outliers ... DBSCAN
  NO, and densities vary a lot ........................... HDBSCAN (separate package)

Special cases
  Millions of rows ....................................... MiniBatchKMeans
  Text / TF-IDF vectors .................................. K-Means on cosine-normalised
                                                            vectors, or NMF/LDA (Notebook 9)
  Need soft/probabilistic membership ..................... GMM
  Non-convex shapes, no outliers ......................... Spectral clustering
  Mixed numeric + categorical ............................ K-Prototypes, or Gower + hierarchical
```

**Universal rules:** scale (and usually log-transform) first; check stability; and never ship
a clustering you cannot describe in words.

In [ ]:
# A head-to-head on four dataset shapes
from sklearn.cluster import MiniBatchKMeans, SpectralClustering

datasets = {
    "blobs": make_blobs(n_samples=350, centers=3, cluster_std=1.0, random_state=1),
    "moons": make_moons(n_samples=350, noise=0.07, random_state=1),
    "circles": make_circles(n_samples=350, noise=0.05, factor=0.45, random_state=1),
    "unequal variance": make_blobs(n_samples=350, centers=3,
                                   cluster_std=[0.4, 1.6, 3.0], random_state=1),
}
algos = {
    "K-Means": lambda X, k: KMeans(k, n_init=10, random_state=0).fit_predict(X),
    "GMM full": lambda X, k: GaussianMixture(k, covariance_type="full", n_init=3,
                                             random_state=0).fit_predict(X),
    "Ward": lambda X, k: AgglomerativeClustering(n_clusters=k).fit_predict(X),
    "DBSCAN": lambda X, k: DBSCAN(eps=0.3, min_samples=5).fit_predict(X),
    "Spectral": lambda X, k: SpectralClustering(k, affinity="nearest_neighbors",
                                                random_state=0).fit_predict(X),
}
fig, axes = plt.subplots(len(datasets), len(algos) + 1, figsize=(19, 13))
for r, (dname, (Xs_, ys_)) in enumerate(datasets.items()):
    Xs_ = StandardScaler().fit_transform(Xs_)
    k_true = len(np.unique(ys_))
    axes[r, 0].scatter(Xs_[:, 0], Xs_[:, 1], c=ys_, cmap="viridis", s=10)
    axes[r, 0].set_ylabel(dname, fontsize=10)
    axes[r, 0].set_title("truth" if r == 0 else "", fontsize=10)
    for c, (aname, fn) in enumerate(algos.items(), start=1):
        lab = fn(Xs_, k_true)
        axes[r, c].scatter(Xs_[:, 0], Xs_[:, 1], c=lab, cmap="viridis", s=10)
        axes[r, c].set_title(f"{aname if r == 0 else ''}\nARI {adjusted_rand_score(ys_, lab):.2f}",
                             fontsize=9)
    for c in range(len(algos) + 1):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
plt.tight_layout(); plt.show()
print("No algorithm wins every row. Match the algorithm to the shape you expect --")
print("and if you have no idea what shape to expect, plot the data first.")

---
## Exercises

**Exercise 1.** Cluster the iris dataset without using the labels. Choose $k$ with the elbow
and silhouette, then check your answer against the true species with ARI. Does the "correct"
$k$ match the number of species?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_iris

iris = load_iris()
Xi, yi = StandardScaler().fit_transform(iris.data), iris.target

rows1 = []
for k in range(2, 9):
    km_ = KMeans(k, n_init=10, random_state=0).fit(Xi)
    rows1.append({"k": k, "inertia": round(km_.inertia_, 2),
                  "silhouette": round(silhouette_score(Xi, km_.labels_), 4),
                  "ARI_vs_species": round(adjusted_rand_score(yi, km_.labels_), 4)})
t1 = pd.DataFrame(rows1)
print(t1.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(t1.k, t1.inertia, "o-", color="steelblue"); ax[0].set_title("Elbow")
ax[1].plot(t1.k, t1.silhouette, "o-", color="seagreen", label="silhouette")
ax[1].plot(t1.k, t1.ARI_vs_species, "o-", color="crimson", label="ARI vs true species")
ax[1].legend(fontsize=8); ax[1].set_title("Silhouette says 2, ARI says 3")
for a_ in ax:
    a_.set_xlabel("k")
plt.tight_layout(); plt.show()

print(f"\nBest silhouette : k = {int(t1.loc[t1.silhouette.idxmax(),'k'])}")
print(f"Best ARI        : k = {int(t1.loc[t1.ARI_vs_species.idxmax(),'k'])}")
print("\nNo, they do not match. Silhouette prefers k=2 because two of the three species")
print("(versicolor and virginica) overlap heavily, so geometrically there are two blobs.")
print("The 'right' k depends on what you are asking: k=2 describes the GEOMETRY honestly,")
print("k=3 matches the BIOLOGY. Unsupervised methods answer the first question, not the")
print("second -- which is exactly why clustering results need domain review.")

**Exercise 2.** Show that DBSCAN's `eps` needs tuning, and use a k-distance plot to pick it
for a dataset you generate with three blobs of different densities plus 5% noise. Report the
number of clusters found and how many points were flagged as noise.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
X2 = np.vstack([
    rng.normal([0, 0], 0.30, (200, 2)),
    rng.normal([3, 3], 0.55, (200, 2)),
    rng.normal([-3, 3], 0.80, (200, 2)),
    rng.uniform(-6, 6, (30, 2)),                       # 5% noise
])
y2 = np.r_[np.zeros(200), np.ones(200), np.full(200, 2), np.full(30, -1)]
X2s = StandardScaler().fit_transform(X2)

kd = np.sort(NearestNeighbors(n_neighbors=8).fit(X2s).kneighbors(X2s)[0][:, -1])
knee_idx = int(np.argmax(np.gradient(np.gradient(kd))))
eps_guess = kd[knee_idx]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(kd, color="steelblue", lw=2)
ax[0].axhline(eps_guess, color="crimson", ls="--", label=f"knee -> eps = {eps_guess:.3f}")
ax[0].set_xlabel("points sorted"); ax[0].set_ylabel("distance to 8th neighbour")
ax[0].set_title("k-distance plot"); ax[0].legend(fontsize=8)

db_best = DBSCAN(eps=eps_guess, min_samples=8).fit(X2s)
ax[1].scatter(X2s[:, 0], X2s[:, 1], c=db_best.labels_, cmap="viridis", s=14)
n_cl = len(set(db_best.labels_)) - (1 if -1 in db_best.labels_ else 0)
ax[1].set_title(f"DBSCAN eps={eps_guess:.3f}: {n_cl} clusters, "
                f"{(db_best.labels_ == -1).sum()} noise\nARI "
                f"{adjusted_rand_score(y2, db_best.labels_):.3f}")
plt.tight_layout(); plt.show()

print(f"{'eps':>7}{'clusters':>10}{'noise':>8}{'ARI':>9}")
for eps in (0.10, 0.20, eps_guess, 0.40, 0.70, 1.2):
    db = DBSCAN(eps=eps, min_samples=8).fit(X2s)
    n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f"{eps:>7.3f}{n_cl:>10}{(db.labels_ == -1).sum():>8}"
          f"{adjusted_rand_score(y2, db.labels_):>9.3f}")
print("\nToo small: everything is noise. Too large: one giant cluster. The knee of the")
print("k-distance curve gets you into the right neighbourhood in one step.")

**Exercise 3.** Compare K-Means and GMM on data with overlapping elliptical clusters. Report
ARI for both, and identify the points where the GMM is genuinely uncertain. Explain when the
soft assignment is worth the extra complexity.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
cov1 = np.array([[2.5, 1.8], [1.8, 1.6]])
cov2 = np.array([[2.0, -1.5], [-1.5, 1.5]])
X3 = np.vstack([rng.multivariate_normal([0, 0], cov1, 300),
                rng.multivariate_normal([2.2, 1.2], cov2, 300)])
y3 = np.r_[np.zeros(300), np.ones(300)]

km3 = KMeans(2, n_init=10, random_state=0).fit(X3)
gm3 = GaussianMixture(2, covariance_type="full", n_init=10, random_state=0).fit(X3)
lab_g = gm3.predict(X3)
conf = gm3.predict_proba(X3).max(axis=1)

print(f"K-Means ARI : {adjusted_rand_score(y3, km3.labels_):.4f}")
print(f"GMM ARI     : {adjusted_rand_score(y3, lab_g):.4f}")
print(f"\nGMM confidence: median {np.median(conf):.3f}, "
      f"{(conf < 0.7).sum()} of {len(conf)} points below 0.7")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].scatter(X3[:, 0], X3[:, 1], c=y3, cmap="coolwarm", s=12)
ax[0].set_title("Truth: two overlapping ellipses")
ax[1].scatter(X3[:, 0], X3[:, 1], c=km3.labels_, cmap="coolwarm", s=12)
ax[1].set_title(f"K-Means (ARI {adjusted_rand_score(y3, km3.labels_):.3f})")
s3 = ax[2].scatter(X3[:, 0], X3[:, 1], c=conf, cmap="RdYlGn", s=14, vmin=0.5, vmax=1)
plt.colorbar(s3, ax=ax[2], label="GMM confidence")
ax[2].set_title(f"GMM (ARI {adjusted_rand_score(y3, lab_g):.3f}); red = uncertain")
plt.tight_layout(); plt.show()

high = conf > 0.9
print(f"\nAmong CONFIDENT points (p > 0.9, {high.sum()} of them), the GMM's partition")
print(f"agrees with the truth at ARI {adjusted_rand_score(y3[high], lab_g[high]):.4f}")
print(f"Among UNCERTAIN points (p <= 0.7), ARI is "
      f"{adjusted_rand_score(y3[conf <= 0.7], lab_g[conf <= 0.7]):.4f}")
print("\nWhen the soft assignment earns its keep:")
print("  * the confidence flags exactly the points where any hard label would be a guess")
print("  * you can act only on high-confidence members (a cleaner marketing list)")
print("  * you can route ambiguous cases to a human or to a second model")
print("  * BIC lets you choose k without an elbow judgement call")
print("Cost: more parameters (k*d*(d+1)/2 covariance terms), slower, and it can")
print("degenerate when a component collapses onto a few points.")

**Exercise 4 (challenge).** You are asked to segment 5,000 customers for a marketing team.
Deliver: a defended choice of $k$, a profile table, a name and action per segment, and an
honest statement of the segmentation's limitations.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
m4 = 5_000
true_kind = rng.choice(["bargain", "mainstream", "premium", "dormant", "wholesale"],
                       m4, p=[0.22, 0.34, 0.18, 0.20, 0.06])
spec = {"bargain":    (18, 40,  9.9, 0.24),
        "mainstream": (12, 90, 10.9, 0.10),
        "premium":    (9, 160, 12.1, 0.05),
        "dormant":    (2, 260, 10.2, 0.12),
        "wholesale":  (30, 25, 13.2, 0.02)}
orders = np.maximum(1, np.array([spec[k_][0] for k_ in true_kind]) + rng.normal(0, 4, m4))
recency = np.array([spec[k_][1] for k_ in true_kind]) * rng.lognormal(0, 0.35, m4)
spend = np.exp(np.array([spec[k_][2] for k_ in true_kind]) + rng.normal(0, 0.45, m4))
ret_rate = np.clip(np.array([spec[k_][3] for k_ in true_kind]) + rng.normal(0, 0.04, m4), 0, 1)
discount_share = np.clip(np.where(true_kind == "bargain", 0.6, 0.15)
                         + rng.normal(0, 0.1, m4), 0, 1)

C = pd.DataFrame({"orders_12m": orders, "recency_days": recency, "total_spend": spend,
                  "return_rate": ret_rate, "discount_share": discount_share})

F = pd.DataFrame({
    "log_orders": np.log1p(C.orders_12m),
    "log_recency": np.log1p(C.recency_days),
    "log_spend": np.log(C.total_spend),
    "log_aov": np.log(C.total_spend / C.orders_12m),
    "return_rate": C.return_rate,
    "discount_share": C.discount_share,
})
Zc = StandardScaler().fit_transform(F)

print("STEP 1 -- choose k, using four signals rather than one\n")
rows4 = []
for k in range(2, 10):
    km_ = KMeans(k, n_init=10, random_state=0).fit(Zc)
    stab, _ = cluster_stability(F.to_numpy(), k, n_boot=12)
    gm_ = GaussianMixture(k, covariance_type="full", n_init=2, random_state=0).fit(Zc)
    rows4.append({"k": k, "silhouette": silhouette_score(Zc, km_.labels_),
                  "davies_bouldin": davies_bouldin_score(Zc, km_.labels_),
                  "stability_ARI": stab, "GMM_BIC": gm_.bic(Zc)})
S = pd.DataFrame(rows4)
print(S.round(4).to_string(index=False))
print(f"\nsilhouette max at k={int(S.loc[S.silhouette.idxmax(),'k'])}, "
      f"Davies-Bouldin min at k={int(S.loc[S.davies_bouldin.idxmin(),'k'])}, "
      f"stability max at k={int(S.loc[S.stability_ARI.idxmax(),'k'])}, "
      f"BIC min at k={int(S.loc[S.GMM_BIC.idxmin(),'k'])}")

In [ ]:
K4 = 5
print(f"DECISION: k = {K4}.")
print("  Justification: it is at or near the optimum for stability and Davies-Bouldin,")
print("  BIC keeps improving past it (BIC will always favour more components when the")
print("  clusters are not exactly Gaussian), and five segments is the largest number the")
print("  marketing team can realistically run distinct campaigns for. When statistical")
print("  and operational answers differ, say so and choose deliberately.\n")

final4 = KMeans(K4, n_init=25, random_state=0).fit(Zc)
C["segment"] = final4.labels_

print("STEP 2 -- the profile table (the actual deliverable)\n")
prof = C.groupby("segment").agg(
    customers=("total_spend", "size"),
    median_orders=("orders_12m", "median"),
    median_recency=("recency_days", "median"),
    median_spend=("total_spend", "median"),
    median_aov=("total_spend", lambda s: np.median(s / C.loc[s.index, "orders_12m"])),
    return_rate=("return_rate", "median"),
    discount_share=("discount_share", "median"),
).round(2)
prof["cust_share_%"] = (prof.customers / len(C) * 100).round(1)
prof["revenue_share_%"] = (C.groupby("segment").total_spend.sum() / C.total_spend.sum()
                           * 100).round(1)
print(prof.to_string())
print(f"\nSanity check against the simulated ground truth: "
      f"ARI = {adjusted_rand_score(true_kind, final4.labels_):.4f}")

In [ ]:
print("STEP 3 -- name each segment and attach one action\n")
for s in range(K4):
    p = prof.loc[s]
    if p.median_aov > 60_000:
        name, action = "WHOLESALE / B2B", "dedicated account manager, volume pricing, invoicing terms"
    elif p.median_recency > 200:
        name, action = "DORMANT", "reactivation offer; suppress from BAU email to protect deliverability"
    elif p.discount_share > 0.4:
        name, action = "BARGAIN HUNTERS", "promote clearance lines only; never discount full-price stock for them"
    elif p.median_spend > np.median(C.total_spend) * 1.6:
        name, action = "PREMIUM", "early access, concierge service, protect margin -- no blanket discounts"
    else:
        name, action = "MAINSTREAM", "cross-sell adjacencies, loyalty points, raise order frequency"
    print(f"  Segment {s}: {name}")
    print(f"    {p['cust_share_%']:.0f}% of customers, {p['revenue_share_%']:.0f}% of revenue, "
          f"median spend {p.median_spend:,.0f}, AOV {p.median_aov:,.0f}")
    print(f"    ACTION: {action}\n")

print("STEP 4 -- limitations, stated up front\n")
print("  1. Segments are DESCRIPTIVE, not causal. They tell you who differs, not what to")
print("     change. Test every action with a holdout group.")
print("  2. The choice of features determines the segments. Adding browsing behaviour or")
print("     product categories would produce a different, equally valid segmentation.")
print(f"  3. Stability is {S.loc[S.k == K4, 'stability_ARI'].item():.2f}, so roughly "
      f"{(1 - S.loc[S.k == K4, 'stability_ARI'].item())*100:.0f}% of pairwise")
print("     co-membership would change on a different sample. Do not treat an individual")
print("     customer's segment as a fact about them.")
print("  4. Boundary customers exist. A GMM's soft probabilities would flag them; a K-Means")
print("     hard label hides them.")
print("  5. Segments DRIFT. Re-fit quarterly, and monitor the share and profile of each")
print("     segment as a data-quality signal.")
print("  6. k=5 is a business constraint as much as a statistical finding. If marketing can")
print("     only run three campaigns, deliver three segments and say what was lost.")

---
## Summary

| Concept | Key point |
|---|---|
| K-Means | Minimise within-cluster squared distance; assign–update until stable |
| Inertia | Always falls with $k$; use it for the elbow, never to choose $k$ alone |
| Assumptions | Spherical, similar-sized, convex clusters; $k$ known; features scaled |
| `k-means++` | Spread-out initialisation; combine with `n_init` restarts |
| Silhouette | $(b-a)/\max(a,b)$; maximise over $k$; read the *plot*, not just the mean |
| Scaling | Mandatory; log-transform skewed monetary features first |
| Hierarchical | Dendrogram, cut where you like; Ward for compact, single for chains |
| DBSCAN | Density-based; finds shapes and noise; very sensitive to `eps` |
| GMM | Soft assignments, elliptical clusters, AIC/BIC for choosing $k$ |
| ARI / NMI | External measures — only available with ground truth |
| Stability | Cluster two overlapping subsamples and compare; the best label-free check |
| The real test | Can you name each cluster and act differently on it? |

**Next up:** [Notebook 6 — Random Forest](6.%20Random%20Forest.ipynb), where averaging many
imperfect trees produces one of the strongest models in practice.